# MATH 3260 Research Project — Starter Notebook 1
## Bulk Google Trends Processing, Lifecycle Alignment, and Time Series as Vectors

This notebook handles the **data preparation and Linear Algebra representation** for the meme research project.

### Folder-first workflow

You do **not** need to open and process 100 CSV files one at a time.

Place all Google Trends downloads in one folder named

`google_trends_csv`

or place all CSV files inside one ZIP file and use the Colab upload cell below.

Then run this notebook from top to bottom. The notebook will automatically:

1. discover every CSV file in the folder;
2. read the search term from the Google Trends column header;
3. convert the history to monthly observations;
4. preserve the **observed peak month** from the original monthly data;
5. apply light smoothing to reduce small month-to-month noise;
6. rescale the **smoothed** curve so its maximum is 100, without applying a second min--max normalization;
7. search backward for the approximate beginning of the major rise;
8. extract a complete 36-month lifecycle vector when possible;
9. separate recent incomplete memes from the full training dataset;
10. stack all complete vectors into one matrix.

> **Important:** Google Trends already reports each downloaded search series on a relative scale whose peak is 100. Therefore, this notebook does **not** apply a second min--max normalization. After smoothing, it only rescales the smoothed curve so that its maximum returns to 100.

> **Important:** The code for the algorithms is supplied. Your responsibility is to understand what each algorithm is doing mathematically and to interpret the results.

> **No future values are invented.** If a recent meme does not yet have 36 observed lifecycle months, the notebook keeps its observed partial trajectory for optional later analysis instead of filling future months with an average, zeros, or repeated values.


## 1. The final mathematical objects

### Complete lifecycle samples

For every meme with a complete lifecycle window, we want

$$
\mathbf{x}_i=
\begin{bmatrix}
x_{i1}\\
x_{i2}\\
\vdots\\
x_{i36}
\end{bmatrix}
\in\mathbb{R}^{36}.
$$

If there are $n$ complete usable memes, stack the vectors as rows:

$$
X=
\begin{bmatrix}
---\mathbf{x}_1^T---\\
---\mathbf{x}_2^T---\\
\vdots\\
---\mathbf{x}_n^T---
\end{bmatrix}
\in\mathbb{R}^{n\times36}.
$$

The project requires at least

$$
\boxed{n\geq100}.
$$

### Recent incomplete samples

A recent meme may have only $m<36$ observed lifecycle months. Then it is a partial vector

$$
\mathbf{z}\in\mathbb{R}^{m}.
$$

It does **not** belong in the $k$-means training matrix $X$ because its dimension differs from 36. The notebook saves these partial curves separately so they can be compared later with the first $m$ components of the learned cluster centroids.

### Before using code

Explain in your own words:

- What does one row of $X$ represent?
- Why must every training row contain exactly 36 entries?
- Why should an incomplete recent meme not be padded with made-up future values?

## 2. Why we align by lifecycle instead of using the latest 36 months

Suppose Meme A became popular in 2014 and Meme B became popular in 2025.

If we simply use the latest 36 months:

- Meme A may contain only a long-term tail;
- Meme B may contain its rise and peak.

Those vectors would represent different lifecycle stages.

Instead, we use the simplified course workflow

$$
\text{long historical series}
\longrightarrow
\text{monthly observations}
\longrightarrow
\text{observed peak}
\longrightarrow
\text{light smoothing}
\longrightarrow
\text{peak rescaling}
\longrightarrow
\text{rise point}
\longrightarrow
\text{lifecycle vector}.
$$

The goal is to compare **lifecycle shapes**, not calendar dates.

Because Google Trends already gives each downloaded search series a relative peak of 100, we do not force the smallest observed value to become 0. This preserves meaningful baseline and tail information.


In [ ]:
from pathlib import Path
import zipfile
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_FOLDER = Path("google_trends_csv")
LIFECYCLE_LENGTH = 36
SMOOTHING_WINDOW = 3
THRESHOLDS = [25, 30, 35, 40, 45, 50]
MIN_PARTIAL_MONTHS_FOR_EXPLORATION = 10

print("Expected extracted-data folder:", DATA_FOLDER)
print("Full lifecycle dimension:", LIFECYCLE_LENGTH)
print("Rise thresholds that may be tried:", THRESHOLDS)


## 3. Upload your Google Trends ZIP file

For this project, the easiest method in **Google Colab** is to upload **one ZIP file** containing all of your Google Trends CSV downloads.

### Before you begin

On your own computer:

1. Put all of your Google Trends CSV files into one folder.
2. Include approximately **110--120 candidate memes** so that you have enough data even if some files cannot be used.
3. Compress the folder into a ZIP file.
4. Give the ZIP file any reasonable name, for example:

```text
animal.zip
```

or

```text
meme_google_trends.zip
```

The name of the ZIP file does **not** matter.

### In Google Colab

Follow these steps exactly:

**Step 1.** Run the next code cell.

**Step 2.** A file-selection window will appear.

**Step 3.** Click **Choose Files** and select the ZIP file from your computer.  
For example, if the file is on your Desktop, select `animal.zip` from the Desktop.

**Step 4.** Wait for the upload to finish. Do not run another cell while the file is still uploading.

**Step 5.** The code will automatically:
- verify that the uploaded file is a ZIP file;
- clear any old extracted Google Trends files from the current Colab session;
- extract all CSV files from the ZIP;
- place them into a folder named `google_trends_csv`;
- count the extracted CSV files.

**Step 6.** Look for a message similar to:

```text
UPLOAD AND EXTRACTION COMPLETE
ZIP file: animal.zip
Number of CSV files extracted: 115
Data folder: google_trends_csv
```

If you see that message, continue to the next section of the notebook.

### Important

You do **not** need to type a Mac or Windows Desktop path into Colab.

For example, do **not** enter a path such as

```text
/Users/yourname/Desktop/animal.zip
```

because Google Colab runs on a remote computer and cannot directly access your Desktop. The upload button transfers the ZIP file from your computer into the Colab session.

### Why begin with 110--120 candidates?

Some real-world search terms may be too sparse, ambiguous, too recent, or may not contain a suitable lifecycle window. Starting with more than 100 candidates makes it much more realistic to finish with at least

$$
100
$$

complete 36-month lifecycle vectors.


In [ ]:
# ============================================================
# STUDENT STEP: Upload ONE ZIP file containing all meme CSVs
# ============================================================

def extract_google_trends_zip(zip_path, data_folder=DATA_FOLDER):
    """
    Extract all CSV files from a ZIP archive into one flat folder.

    The ZIP may contain:
      - CSV files directly, or
      - CSV files inside one or more subfolders.

    Only .csv files are extracted.
    """
    zip_path = Path(zip_path)
    data_folder = Path(data_folder)

    # Start fresh so files from an earlier upload are not mixed
    # with the current student's dataset.
    if data_folder.exists():
        shutil.rmtree(data_folder)

    data_folder.mkdir(parents=True, exist_ok=True)
    extracted = []

    with zipfile.ZipFile(zip_path, "r") as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)

            # Skip folders, non-CSV files, and common macOS metadata files.
            if member.is_dir():
                continue
            if member_path.suffix.lower() != ".csv":
                continue
            if member_path.name.startswith("._"):
                continue

            # Put every CSV into one flat folder.
            target = data_folder / member_path.name

            # Avoid overwriting files if two CSVs have the same filename.
            if target.exists():
                stem = target.stem
                suffix = target.suffix
                counter = 2

                while target.exists():
                    target = data_folder / f"{stem}_{counter}{suffix}"
                    counter += 1

            target.write_bytes(archive.read(member))
            extracted.append(target)

    return extracted


# ------------------------------------------------------------
# Google Colab upload
# ------------------------------------------------------------
try:
    from google.colab import files
except ImportError:
    raise RuntimeError(
        "This upload cell is designed for Google Colab. "
        "Open the notebook in Colab and run this cell there."
    )

print("=" * 65)
print("UPLOAD YOUR GOOGLE TRENDS ZIP FILE")
print("=" * 65)
print()
print("1. Click 'Choose Files' below.")
print("2. Select ONE .zip file containing all Google Trends CSV files.")
print("3. Wait until the upload finishes.")
print()

uploaded = files.upload()

# Keep only uploaded ZIP files.
zip_names = [
    name for name in uploaded.keys()
    if str(name).lower().endswith(".zip")
]

if len(zip_names) == 0:
    raise ValueError(
        "No ZIP file was uploaded. Please rerun this cell and select one .zip file."
    )

if len(zip_names) > 1:
    raise ValueError(
        "More than one ZIP file was uploaded. "
        "Please rerun this cell and select only ONE ZIP file."
    )

selected_zip = Path(zip_names[0])

if not zipfile.is_zipfile(selected_zip):
    raise ValueError(
        f"'{selected_zip.name}' does not appear to be a valid ZIP file. "
        "Please create a ZIP archive containing your CSV files and try again."
    )

# Extract the CSV files.
extracted_files = extract_google_trends_zip(
    selected_zip,
    DATA_FOLDER
)

# Verify the result.
csv_files = sorted(DATA_FOLDER.glob("*.csv"))

if len(csv_files) == 0:
    raise ValueError(
        "The ZIP file was uploaded successfully, but no CSV files were found inside it."
    )

print()
print("=" * 65)
print("UPLOAD AND EXTRACTION COMPLETE")
print("=" * 65)
print("ZIP file:", selected_zip.name)
print("Number of CSV files extracted:", len(csv_files))
print("Data folder:", DATA_FOLDER)
print()

print("First few CSV files:")
for path in csv_files[:10]:
    print(" -", path.name)

if len(csv_files) < 100:
    print()
    print("WARNING:")
    print(
        f"Only {len(csv_files)} CSV files were extracted. "
        "The project requires at least 100 complete usable lifecycle vectors, "
        "so you will probably need additional candidate memes."
    )
else:
    print()
    print("The upload step is complete. Continue to the next section of the notebook.")


### Quick check before continuing

Before moving on, confirm that the output above shows:

- the correct ZIP filename;
- a positive number of extracted CSV files;
- the folder `google_trends_csv`.

You do **not** need to open each CSV file manually. The remainder of this notebook will read and process the entire folder automatically.

If you uploaded the wrong ZIP file, simply rerun the upload cell and choose the correct file. The old extracted folder will be cleared automatically.


## 4. Algorithm A — Read one Google Trends CSV

The reader performs several tasks automatically:

1. detect whether the file is comma-separated or tab-separated;
2. identify the date/time column;
3. identify the Google Trends interest column;
4. remove a row marked `isPartial=True` when the export contains an incomplete current month;
5. convert the dates to a time index;
6. convert interest values to numbers.

This flexibility is useful because some Google Trends data files use a `.csv` filename even when the values are actually separated by tabs.

Google Trends sometimes represents very small values as `<1`. For this course notebook, `<1` is converted to $0.5$ so that it remains a small positive value.

The column header of the interest column is also returned as the **search term**. This is what allows the notebook to discover a whole folder automatically.


In [ ]:
def read_google_trends_csv(path):
    """
    Read one Google Trends export.

    The function is intentionally flexible because files saved with a .csv
    extension may be comma-separated OR tab-separated. It also ignores a
    Google Trends 'isPartial' row when that row represents an incomplete
    current month.
    """
    path = Path(path)

    def find_date_column(columns):
        for c in columns:
            name = str(c).lower()
            if any(key in name for key in ["week", "month", "day", "date", "time"]):
                return c
        return None

    def read_table(skiprows=0):
        # sep=None with the Python engine asks pandas to detect the delimiter.
        # This handles both ordinary comma-separated Google Trends files
        # and tab-separated files such as the animal.zip dataset.
        return pd.read_csv(
            path,
            sep=None,
            engine="python",
            skiprows=skiprows,
        )

    # First try the ordinary header row.
    df = read_table(skiprows=0)
    date_col = find_date_column(df.columns)

    # Some Google Trends exports include one descriptive line above the header.
    if date_col is None:
        df = read_table(skiprows=1)
        date_col = find_date_column(df.columns)

    if date_col is None:
        raise ValueError("Could not identify a date/time column.")

    # If the export includes an isPartial column, remove rows marked True.
    # Those rows represent an incomplete current reporting period.
    partial_columns = [
        c for c in df.columns
        if "partial" in str(c).lower()
    ]

    if partial_columns:
        partial_col = partial_columns[0]
        partial_mask = (
            df[partial_col]
            .astype(str)
            .str.strip()
            .str.lower()
            .isin(["true", "1", "yes"])
        )
        df = df.loc[~partial_mask].copy()

    value_candidates = [
        c for c in df.columns
        if c != date_col and "partial" not in str(c).lower()
    ]

    if not value_candidates:
        raise ValueError("Could not identify a Google Trends interest column.")

    value_col = value_candidates[0]

    dates = pd.to_datetime(df[date_col], errors="coerce")

    raw_values = (
        df[value_col]
        .astype(str)
        .str.replace("<1", "0.5", regex=False)
        .str.replace(",", "", regex=False)
    )

    values = pd.to_numeric(raw_values, errors="coerce")

    series = pd.Series(
        values.to_numpy(),
        index=dates,
        name=str(value_col),
    )

    series = series[~series.index.isna()].dropna().sort_index()
    series = series[~series.index.duplicated(keep="first")]

    if len(series) == 0:
        raise ValueError("No usable numerical observations were found.")

    return {
        "series": series,
        "search_term": str(value_col),
        "date_column": str(date_col),
        "value_column": str(value_col),
        "file_name": path.name,
    }


## 5. Algorithm B — Automatically discover the entire folder

The folder scanner applies Algorithm A to every `*.csv` file.

For each file it records:

- file name;
- search term from the column header;
- first observed date;
- last observed date;
- number of raw observations;
- whether the file could be read.

This gives a first metadata table without requiring the student to type 100 filenames manually.

You may later add a friendlier `meme_name` or notes if a Google Trends search term is abbreviated or ambiguous.

In [ ]:
def discover_google_trends_folder(data_folder):
    data_folder = Path(data_folder)
    records = []

    if not data_folder.exists():
        return pd.DataFrame()

    csv_files = sorted(data_folder.glob("*.csv"))

    for candidate_id, path in enumerate(csv_files, start=1):
        try:
            parsed = read_google_trends_csv(path)
            s = parsed["series"]
            records.append({
                "candidate_id": candidate_id,
                "file_name": path.name,
                "meme_name": parsed["search_term"],
                "search_term": parsed["search_term"],
                "first_observed_date": str(s.index.min().date()),
                "last_observed_date": str(s.index.max().date()),
                "raw_observation_count": len(s),
                "folder_read_status": "read",
                "notes": "",
            })
        except Exception as exc:
            records.append({
                "candidate_id": candidate_id,
                "file_name": path.name,
                "meme_name": path.stem,
                "search_term": "",
                "first_observed_date": "",
                "last_observed_date": "",
                "raw_observation_count": 0,
                "folder_read_status": "error",
                "notes": str(exc),
            })

    return pd.DataFrame(records)

candidate_table = discover_google_trends_folder(DATA_FOLDER)

if len(candidate_table):
    print("CSV files discovered:", len(candidate_table))
    display(candidate_table.head(10))
else:
    print("No CSV files found yet. Create the folder 'google_trends_csv' and place all downloads inside it.")


## 6. Algorithm C — Convert to monthly observations

Google Trends may return weekly or monthly observations depending on the selected historical range.

Our final vector coordinates must have a common time meaning, so every history is converted to **monthly observations**.

If a month contains observations

$$
y_1,y_2,\ldots,y_m,
$$

we use the monthly mean

$$
\overline{y}=\frac{1}{m}\sum_{j=1}^{m}y_j.
$$

### Missing months

The notebook will interpolate only **short internal gaps of at most two months**. A long missing block is not silently filled because that would manufacture too much of the trajectory.

This is different from missing **future** months. Future months are never interpolated because they have not happened yet.

In [ ]:
def to_monthly(series):
    monthly = series.resample("MS").mean()

    # Only fill short gaps inside the observed time range.
    monthly = monthly.interpolate(
        method="linear",
        limit=2,
        limit_area="inside",
    )

    if monthly.isna().any():
        missing = int(monthly.isna().sum())
        raise ValueError(
            f"The monthly series contains {missing} unresolved missing months; review this candidate."
        )

    return monthly


## 7. Algorithm D — Identify the observed peak from the original monthly data

The peak month should describe what was actually observed, not a peak shifted by smoothing.

If the monthly values are

$$
y_1,y_2,\ldots,y_T,
$$

we define the observed peak position by

$$
p=\operatorname*{arg\,max}_{1\le t\le T}y_t.
$$

`arg max` returns the **position** of the largest value.

If several months tie for the maximum, the notebook uses the first one.

### Why detect the peak before smoothing?

A centered moving average may move the numerical maximum one month earlier or later. For reporting, we want the actual Google Trends peak month. Smoothing is used only to make the **rise-detection trend** less noisy.

In [ ]:
def find_observed_peak(monthly):
    values = monthly.to_numpy(dtype=float)
    peak_pos = int(np.argmax(values))
    return {
        "peak_pos": peak_pos,
        "peak_month": monthly.index[peak_pos],
        "peak_value": float(values[peak_pos]),
    }


## 8. Algorithm E — Light smoothing

A three-month centered moving average is approximately

$$
\widetilde{x}_t
=
\frac{x_{t-1}+x_t+x_{t+1}}{3}.
$$

The purpose is to reduce small month-to-month noise before locating the major rise and comparing lifecycle shapes.

We do **not** use heavy smoothing because repeated spikes or resurgences may be meaningful lifecycle features.

The observed peak month is still determined from the original monthly Google Trends data **before** smoothing, because smoothing can shift the location of the numerical maximum.


In [ ]:
def smooth_monthly(series, window=3):
    if window <= 1:
        return series.copy()
    return series.rolling(window=window, center=True, min_periods=1).mean()


## 9. Algorithm F — Peak-rescale the smoothed curve

Google Trends already reports each downloaded search series on a relative scale with

$$
100=\text{the observed peak interest for that search term under the selected settings}.
$$

Therefore, we do **not** apply min--max normalization a second time.

Why not? Min--max normalization would force the smallest observed value to 0:

$$
100\frac{x-x_{\min}}{x_{\max}-x_{\min}}.
$$

That can distort meaningful baseline or tail interest.

However, smoothing usually lowers the numerical maximum. For example, a raw peak of 100 may become 82 after a three-month moving average.

So, after smoothing, we use only **peak rescaling**:

$$
x_t^{(\mathrm{peak})}
=
100\frac{\widetilde{x}_t}{\max_s \widetilde{x}_s}.
$$

This restores the maximum of the smoothed curve to 100 while preserving the relative baseline.

### Example

If the smoothed values are

$$
[15,25,40,82,60,35],
$$

then peak rescaling gives approximately

$$
[18.3,30.5,48.8,100,73.2,42.7].
$$

Notice that the first value remains positive. We do not artificially force it to 0.

If the smoothed curve has no positive peak, the candidate is rejected because it does not contain a usable popularity trajectory.


In [ ]:
def rescale_peak_to_100(series):
    values = series.to_numpy(dtype=float)
    vmax = float(np.max(values))

    if not np.isfinite(vmax) or vmax <= 0:
        raise ValueError("The smoothed series has no positive peak and cannot be peak-rescaled.")

    scaled = 100.0 * values / vmax
    return pd.Series(scaled, index=series.index, name="peak_rescaled_trend")


## 10. Algorithm G — Find the approximate beginning of the major rise

We keep the **observed peak position** fixed, then search backward through the smoothed, peak-rescaled trend.

Start with

$$
q=25.
$$

Among the months before the observed peak, find the closest earlier month satisfying

$$
x_s\le q.
$$

If no such point exists, try

$$
30,35,40,45,50.
$$

The first threshold that finds a pre-peak point is used.

### Worked example

Suppose the peak-rescaled trend before and around the peak contains

$$
[8,12,18,24,33,48,67,84,100].
$$

Starting at the peak and moving backward:

$$
84>25,
$$

$$
67>25,
$$

$$
48>25,
$$

$$
33>25,
$$

but

$$
24\le25.
$$

Therefore the component containing 24 is selected as the approximate rise point.

This is a simplified course-level rule. It is not intended to reproduce every detail of the published research algorithm.


In [ ]:
def find_rise_start(peak_rescaled_trend, peak_pos, thresholds=(25, 30, 35, 40, 45, 50)):
    values = peak_rescaled_trend.to_numpy(dtype=float)

    if peak_pos <= 0:
        return {
            "success": False,
            "reason": "The observed peak occurs at the beginning of the available history.",
            "threshold": None,
            "start_pos": None,
            "start_month": None,
        }

    pre_peak = values[:peak_pos]

    for threshold in thresholds:
        candidates = np.where(pre_peak <= threshold)[0]
        if len(candidates) > 0:
            start_pos = int(candidates[-1])
            return {
                "success": True,
                "reason": "rise point found",
                "threshold": threshold,
                "start_pos": start_pos,
                "start_month": peak_rescaled_trend.index[start_pos],
            }

    return {
        "success": False,
        "reason": "No suitable pre-peak rise point was found using thresholds 25 through 50.",
        "threshold": None,
        "start_pos": None,
        "start_month": None,
    }


## 11. Algorithm H — Extract a complete 36-month lifecycle when it exists

If the detected rise position is $s$, the desired full vector is

$$
\mathbf{x}
=
[x_s,x_{s+1},\ldots,x_{s+35}]^T
\in\mathbb{R}^{36}.
$$

There are now three possible outcomes.

### Outcome 1 — `use_full`

At least 36 observed months are available from the rise point. The meme contributes one row to the training matrix $X$.

### Outcome 2 — `partial_recent`

A meaningful rise was found, but fewer than 36 observed months exist afterward because the meme is recent.

The notebook keeps the observed portion

$$
\mathbf{z}
=
[x_s,x_{s+1},\ldots,x_{s+m-1}]^T
\in\mathbb{R}^{m},
m<36,
$$

but does **not** put it into $X$.

### Outcome 3 — `replace`

The curve has no usable pre-peak rise or another serious data-quality problem.

### Why we do not fill missing future months with an average

Suppose a recent meme has 14 observed lifecycle months. Filling Months 15--36 with its recent average would impose an artificial flat tail. Repeating the last value would do the same. Filling with zero would impose an artificial decay.

Any of those choices could determine the cluster assignment before the future is observed.

Therefore, missing future months are treated as **unknown**, not as missing values to be imputed.

In [ ]:
def extract_lifecycle_window(peak_rescaled_trend, peak_info, length=36, thresholds=(25, 30, 35, 40, 45, 50)):
    rise = find_rise_start(
        peak_rescaled_trend,
        peak_pos=peak_info["peak_pos"],
        thresholds=thresholds,
    )

    base = {
        **peak_info,
        **rise,
    }

    if not rise["success"]:
        return None, None, {
            **base,
            "status": "replace",
            "months_available_after_rise": 0,
            "months_missing_for_36": length,
            "analysis_end_month": None,
        }

    start = rise["start_pos"]
    available = len(peak_rescaled_trend) - start

    if available >= length:
        full = peak_rescaled_trend.iloc[start:start + length].copy()
        return full, None, {
            **base,
            "status": "use_full",
            "reason": "complete 36-month lifecycle window extracted",
            "months_available_after_rise": available,
            "months_missing_for_36": 0,
            "analysis_end_month": full.index[-1],
        }

    partial = peak_rescaled_trend.iloc[start:].copy()
    return None, partial, {
        **base,
        "status": "partial_recent",
        "reason": f"Only {available} observed lifecycle months are currently available; {length} are required for the training matrix.",
        "months_available_after_rise": available,
        "months_missing_for_36": length - available,
        "analysis_end_month": partial.index[-1] if len(partial) else None,
    }


## 12. One preprocessing function for one file

This wrapper combines the algorithms already explained.

Notice the distinction:

- `monthly` is the observed monthly Google Trends history;
- `peak_info` comes from the original monthly history;
- `smoothed` is the lightly smoothed monthly curve;
- `peak_rescaled_trend` is the smoothed curve rescaled so that its maximum is 100;
- `full_lifecycle` contains exactly 36 peak-rescaled values only when all 36 months are observed;
- `partial_lifecycle` contains the observed lifecycle so far for a recent incomplete meme.

There is **no second min--max normalization** in this pipeline.


In [ ]:
def preprocess_meme_csv(path, smoothing_window=3, lifecycle_length=36):
    parsed = read_google_trends_csv(path)
    monthly = to_monthly(parsed["series"])
    peak_info = find_observed_peak(monthly)

    smoothed = smooth_monthly(monthly, window=smoothing_window)
    peak_rescaled_trend = rescale_peak_to_100(smoothed)

    full_lifecycle, partial_lifecycle, info = extract_lifecycle_window(
        peak_rescaled_trend,
        peak_info=peak_info,
        length=lifecycle_length,
        thresholds=THRESHOLDS,
    )

    return {
        "file_name": parsed["file_name"],
        "search_term": parsed["search_term"],
        "raw": parsed["series"],
        "monthly": monthly,
        "smoothed": smoothed,
        "peak_rescaled_trend": peak_rescaled_trend,
        "full_lifecycle": full_lifecycle,
        "partial_lifecycle": partial_lifecycle,
        "info": info,
    }


## 13. Diagnostic plot for any individual file

The plot marks:

- the complete **smoothed, peak-rescaled trend**;
- the **observed Google Trends peak month** from the original monthly data;
- the detected rise point;
- the full 36-month region if available;
- or the observed partial lifecycle if the meme is too recent.

This is an important quality-control step. An algorithm can follow its rule correctly even when a search term is scientifically questionable.


In [ ]:
def plot_lifecycle_diagnostic(result, title=None):
    trend = result["peak_rescaled_trend"]
    info = result["info"]
    title = title or result.get("search_term", "Lifecycle diagnostic")

    plt.figure(figsize=(10, 4.5))
    plt.plot(trend.index, trend.values, label="Smoothed, peak-rescaled trend")

    peak_month = info.get("peak_month")
    if peak_month is not None:
        plt.axvline(peak_month, linestyle="--", label="Observed peak month")

    start_month = info.get("start_month")
    if start_month is not None:
        plt.axvline(start_month, linestyle=":", label="Detected rise point")

    if result["full_lifecycle"] is not None:
        w = result["full_lifecycle"]
        plt.axvspan(w.index[0], w.index[-1], alpha=0.18, label="Complete 36-month lifecycle")
    elif result["partial_lifecycle"] is not None:
        w = result["partial_lifecycle"]
        plt.axvspan(w.index[0], w.index[-1], alpha=0.12, label="Observed partial lifecycle")

    plt.xlabel("Calendar month")
    plt.ylabel("Peak-rescaled Google Trends interest")
    plt.title(title)
    plt.legend()
    plt.show()


## 14. Process the entire folder at once

This is the main batch-processing algorithm.

For every CSV file, it:

1. runs the complete preprocessing pipeline;
2. records a quality-control row;
3. appends a complete 36-dimensional vector to the training list when status is `use_full`;
4. stores recent incomplete vectors in a separate long-format table when status is `partial_recent`.

Finally,

```python
X = np.vstack(full_vectors)
```

stacks the complete row vectors into one matrix.

If there are $n$ complete vectors, then

$$
X\in\mathbb{R}^{n\times36}.
$$

### Why `np.vstack` matches the mathematics

If

$$
\mathbf{x}_1^T,\mathbf{x}_2^T,\ldots,\mathbf{x}_n^T
$$

are row vectors, vertical stacking constructs

$$
\begin{bmatrix}
\mathbf{x}_1^T\\
\mathbf{x}_2^T\\
\vdots\\
\mathbf{x}_n^T
\end{bmatrix}.
$$

So this programming command is exactly the matrix construction described in the course.

In [ ]:
def process_entire_folder(data_folder):
    data_folder = Path(data_folder)
    csv_files = sorted(data_folder.glob("*.csv")) if data_folder.exists() else []

    full_vectors = []
    full_records = []
    qc_records = []
    partial_records = []

    for candidate_id, path in enumerate(csv_files, start=1):
        try:
            result = preprocess_meme_csv(path)
            info = result["info"]
            search_term = result["search_term"]

            qc = {
                "candidate_id": candidate_id,
                "file_name": path.name,
                "meme_name": search_term,
                "search_term": search_term,
                "first_observed_month": str(result["monthly"].index.min().date()),
                "last_observed_month": str(result["monthly"].index.max().date()),
                "observed_peak_month": str(info.get("peak_month", ""))[:10],
                "observed_peak_value": info.get("peak_value", ""),
                "rise_threshold": info.get("threshold", ""),
                "rise_month": str(info.get("start_month", ""))[:10],
                "analysis_end_month": str(info.get("analysis_end_month", ""))[:10],
                "months_available_after_rise": info.get("months_available_after_rise", ""),
                "months_missing_for_36": info.get("months_missing_for_36", ""),
                "usable_status": info.get("status", "review"),
                "reason": info.get("reason", ""),
                "notes": "",
            }
            qc_records.append(qc)

            if result["full_lifecycle"] is not None:
                full_vectors.append(result["full_lifecycle"].to_numpy(dtype=float))
                full_records.append(qc)

            if result["partial_lifecycle"] is not None:
                partial = result["partial_lifecycle"]
                for relative_month, (calendar_month, value) in enumerate(partial.items(), start=1):
                    partial_records.append({
                        "candidate_id": candidate_id,
                        "file_name": path.name,
                        "meme_name": search_term,
                        "search_term": search_term,
                        "lifecycle_month": relative_month,
                        "calendar_month": str(calendar_month.date()),
                        "peak_rescaled_interest": float(value),
                        # Compatibility alias used by Starter Notebook 2.
                        "normalized_interest": float(value),
                    })

        except Exception as exc:
            qc_records.append({
                "candidate_id": candidate_id,
                "file_name": path.name,
                "meme_name": path.stem,
                "search_term": "",
                "usable_status": "error",
                "reason": str(exc),
                "notes": "",
            })

    X = np.vstack(full_vectors) if full_vectors else np.empty((0, LIFECYCLE_LENGTH))
    full_metadata = pd.DataFrame(full_records)
    quality_control = pd.DataFrame(qc_records)
    partial_long = pd.DataFrame(partial_records)

    return X, full_metadata, quality_control, partial_long


In [ ]:
X, usable_metadata, quality_control, partial_recent = process_entire_folder(DATA_FOLDER)

print("Complete 36-month training matrix shape:", X.shape)
print("Complete usable memes:", X.shape[0])
print("Required minimum for the main project:", 100)

if len(quality_control):
    status_counts = quality_control["usable_status"].value_counts(dropna=False)
    print("\nQuality-control status counts:")
    display(status_counts.to_frame("count"))

    display_columns = [
        "candidate_id", "meme_name", "observed_peak_month", "rise_threshold",
        "rise_month", "months_available_after_rise", "months_missing_for_36",
        "usable_status", "reason"
    ]
    display(quality_control[[c for c in display_columns if c in quality_control.columns]].head(20))
else:
    print("No files have been processed yet.")


## 15. Inspect a specific meme after batch processing

If you want to inspect one candidate, give its CSV filename below.

For example, if one downloaded file contains the search term `67`, the diagnostic plot can show:

- the long historical Google Trends record;
- the observed peak;
- the detected rise;
- whether 36 post-rise months actually exist.

A very recent meme may be labeled `partial_recent`. That is not a computational error; it means the future portion of its lifecycle has not yet been observed.

In [ ]:
# Example:
# file_to_inspect = DATA_FOLDER / "your_downloaded_file.csv"
# result = preprocess_meme_csv(file_to_inspect)
# print(result["info"])
# plot_lifecycle_diagnostic(result, title=result["search_term"])


## 16. Save and download the processed data for Notebook 2

At this stage, Notebook 1 has finished the main preprocessing.

The notebook first saves up to three CSV files:

### A. `meme_vectors_36_month.csv`

This is the most important file. It contains only complete 36-month lifecycle vectors and supplies the matrix

$$
X\in\mathbb{R}^{n\times36}
$$

for Notebook 2.

### B. `meme_quality_control.csv`

This records what happened to every candidate meme: complete, recent/incomplete, replace, or processing problem.

### C. `meme_partial_recent_long.csv`

This stores observed portions of recent incomplete meme lifecycles when such cases exist.

### One-file handoff to Notebook 2

To make the workflow easier, the next cell packages all available processed files into

```text
MATH3260_Meme_Project_Processed_Data.zip
```

and, in Google Colab, automatically starts a download of that ZIP file.

**Student steps**

1. Run the next code cell.
2. Wait until you see `PROCESSED DATA PACKAGE READY`.
3. Your browser should download `MATH3260_Meme_Project_Processed_Data.zip`.
4. Save that ZIP file somewhere you can find it, such as your Downloads folder.
5. **Do not unzip it manually.**
6. Open Starter Notebook 2.
7. Notebook 2 will ask you to upload this same ZIP file.

This creates a smooth handoff:

```text
Notebook 1
    ↓
MATH3260_Meme_Project_Processed_Data.zip
    ↓
Notebook 2
```


In [ ]:
# ============================================================
# SAVE PROCESSED FILES AND CREATE ONE ZIP FOR NOTEBOOK 2
# ============================================================

OUTPUT_FILES = []

# 1. Save the complete 36-month vector dataset.
if X.shape[0] > 0:
    vector_columns = [f"t{j:02d}" for j in range(1, LIFECYCLE_LENGTH + 1)]
    vector_df = pd.DataFrame(X, columns=vector_columns)
    cleaned = pd.concat(
        [usable_metadata.reset_index(drop=True), vector_df],
        axis=1
    )

    vector_path = Path("meme_vectors_36_month.csv")
    cleaned.to_csv(vector_path, index=False)
    OUTPUT_FILES.append(vector_path)
    print("Saved:", vector_path)

# 2. Save the quality-control table.
if len(quality_control):
    quality_path = Path("meme_quality_control.csv")
    quality_control.to_csv(quality_path, index=False)
    OUTPUT_FILES.append(quality_path)
    print("Saved:", quality_path)

# 3. Save recent partial lifecycles, when present.
if len(partial_recent):
    partial_path = Path("meme_partial_recent_long.csv")
    partial_recent.to_csv(partial_path, index=False)
    OUTPUT_FILES.append(partial_path)
    print("Saved:", partial_path)

# The full-vector file is required for Notebook 2.
required_file = Path("meme_vectors_36_month.csv")

if not required_file.exists():
    raise ValueError(
        "The required file 'meme_vectors_36_month.csv' was not created. "
        "Check the quality-control results before continuing."
    )

# 4. Package all available processed files into ONE ZIP file.
PACKAGE_FILE = Path("MATH3260_Meme_Project_Processed_Data.zip")

if PACKAGE_FILE.exists():
    PACKAGE_FILE.unlink()

with zipfile.ZipFile(PACKAGE_FILE, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT_FILES:
        if path.exists():
            archive.write(path, arcname=path.name)

print()
print("=" * 68)
print("PROCESSED DATA PACKAGE READY")
print("=" * 68)
print("Package:", PACKAGE_FILE.name)
print("Complete lifecycle vectors:", X.shape[0])
print("Vector dimension:", X.shape[1] if X.ndim == 2 else "N/A")
print("Files included:")
for path in OUTPUT_FILES:
    if path.exists():
        print(" -", path.name)

# 5. Automatically download the ZIP in Google Colab.
try:
    from google.colab import files

    print()
    print("Your download should begin automatically.")
    print("Keep this ZIP file for Starter Notebook 2.")
    files.download(str(PACKAGE_FILE))

except ImportError:
    print()
    print("Automatic browser download is available in Google Colab.")
    print("The ZIP file has been created at:")
    print(PACKAGE_FILE.resolve())


## 17. Optional: why partial recent curves can still be scientifically useful

Suppose a recent meme has only

$$
m=14
$$

observed lifecycle months.

We should not create fake values for Months 15--36. But after the complete historical memes have been clustered in Notebook 2, we can compare the partial recent vector

$$
\mathbf{z}\in\mathbb{R}^{14}
$$

with the first 14 components of each centroid

$$
\boldsymbol{\mu}_{j,1:14}\in\mathbb{R}^{14}.
$$

For example,

$$
d_j^{(14)}
=
\left\|
\mathbf{z}
-
\boldsymbol{\mu}_{j,1:14}
\right\|_2.
$$

This answers a limited but meaningful question:

> Based on the lifecycle observed **so far**, which mature historical pattern does this recent meme most resemble?

It does **not** claim to know the meme's actual future values.

## 18. Required reflection before moving to Notebook 2

Write a short paragraph addressing each question.

1. Why can all downloaded CSV files be processed together even though their calendar dates differ?
2. Why is the observed peak detected before smoothing?
3. Google Trends already uses a relative peak scale. Why do we avoid a second min--max normalization, and why do we peak-rescale after smoothing?
4. How does the threshold search identify an approximate rise point?
5. Why must the main clustering matrix contain only complete 36-dimensional vectors?
6. Why would filling unknown future months with a previous average bias the analysis?
7. What is the difference between a `partial_recent` meme and a truly unusable meme?
8. If the final complete dataset contains 107 memes, what are the dimensions of $X$?

The explanations are more important to this course than memorizing the Python syntax.
